In [1]:
from agents_src.utils import get_llm

prompt = "What is Agentic AI?"
llm = get_llm()
response = llm.invoke(prompt)
print(response.content)

**Agentic AI** – a term that has emerged in recent AI research and policy discussions – refers to artificial intelligence systems that possess *agency*: the capacity to set, pursue, and adapt goals, make decisions, and act independently of direct human instruction. In other words, an agentic AI is not just a tool that follows pre‑programmed rules; it behaves like an autonomous “agent” that can reason about its environment, plan actions, and modify its behavior based on experience.

---

## 1. Core Characteristics

| Feature | What it Means | Typical Example |
|---------|---------------|-----------------|
| **Goal‑oriented behavior** | The system has a clear objective or set of objectives it seeks to achieve. | A delivery drone that aims to deliver packages efficiently. |
| **Autonomous decision‑making** | It can choose actions without explicit human commands at every step. | A self‑driving car deciding how to navigate traffic. |
| **Learning & adaptation** | It improves its performance

In [1]:
import pandas as pd

data = pd.read_csv("agents_src/agent/mock_products.csv")

In [3]:
data.head(5)

,product_id,product_name,category,description,price_inr,status,pros,cons
0,RZ01,Razer DeathAdder V3 Pro,Mouse,Ergonomic wireless gaming mouse featuring a 30...,149.99,Active,Extremely lightweight; Excellent sensor perfor...,No RGB lighting; Only for right-handed users.
1,RZ02,Razer Viper V2 Pro,Mouse,Symmetrical ultra-lightweight (58g) wireless e...,149.99,Active,Flawless wireless tracking; Ambidextrous shape...,Lacks side grips out of the box; No RGB lighting.
2,RZ03,Razer Basilisk V3 Pro,Mouse,Customizable wireless gaming mouse featuring 1...,159.99,Active,Highly customizable buttons; Infinite scroll w...,Heavy compared to esports mice; Higher price p...
3,RZ04,Razer Huntsman V2,Keyboard,Full-sized optical gaming keyboard featuring R...,199.99,Active,Zero-latency optical switches; Comfortable wri...,Linear switches can be noisy; Cable is not det...
4,RZ05,Razer BlackWidow V4 Pro,Keyboard,Premium mechanical gaming keyboard featuring g...,229.99,Active,Dedicated macro keys and command dial; Bright ...,Takes up a lot of desk space; Expensive for a ...


In [1]:
import os
from dotenv import load_dotenv

# Load environment variables (like GROQ_API_KEY)
load_dotenv()

from agents_src.agent.support_agent import SupportAgent

def test_product_detection():
    print("Initializing SupportAgent...")
    agent = SupportAgent()
    
    test_queries = [
        "I need help with my Razer DeathAdder V3 Pro and my Razer BlackWidow V4 Pro, they are not connecting.",
        "How do I clean my mouse?",
        "Is the Razer Blade 16 good for video editing?"
    ]
    
    for query in test_queries:
        print(f"\n--- Testing Query: '{query}' ---")
        
        # Setup initial state
        state = {
            "user_query": query,
            "session_id": "test_session",
            "product_info": {},
            "support_response": "",
            "extracted_product_names": []
        }
        
        # Directly call the product detection method (testing it in isolation)
        result_state = agent._product_detection(state)
        
        # We are fetching 'extracted_product_names' which matches the AgentState exactly
        print("Detected Products:", result_state.get("extracted_product_names"))

test_product_detection()


Initializing SupportAgent...

--- Testing Query: 'I need help with my Razer DeathAdder V3 Pro and my Razer BlackWidow V4 Pro, they are not connecting.' ---
Detected Products: ['Razer DeathAdder V3 Pro', 'Razer BlackWidow V4 Pro']

--- Testing Query: 'How do I clean my mouse?' ---
Detected Products: ['mouse']

--- Testing Query: 'Is the Razer Blade 16 good for video editing?' ---
Detected Products: ['Razer Blade 16']


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from agents_src.agent.support_agent import SupportAgent

def test_product_retrieval():
    print("Initializing SupportAgent...")
    agent = SupportAgent()
    
    # Simulating the state as if the detection node has already run
    state = {
        "user_query": "I need a mouse, a keyboard, the Razer BlackShark V2 Pro, and a Fake Product.",
        "session_id": "test_session",
        "product_info": {},
        "support_response": "",
        "extracted_product_names": [
            "mouse",                   # Category match (should return 2 mice)
            "keyboard",                # Category match (should return 2 keyboards)
            "Razer BlackShark V2 Pro", # Exact product match
            "FakeRazer 9000",          # Fake product
            "Razer Basilisk V3 Pro"
        ]
    }
    
    print(f"\n--- Testing Retrieval for: {state['extracted_product_names']} ---\n")
    
    # Directly call the retrieval method
    result_state = agent._retrieve_product_info(state)
    
    # Extract the resulting product info list
    products_found = result_state.get("product_info", {}).get("products", [])
    
    # Print out the retrieved information nicely
    for idx, product in enumerate(products_found, 1):
        print(f"Product #{idx}:")
        for key, value in product.items():
            print(f"  - {key}: {value}")
        print("-" * 50)
            
test_product_retrieval()


Initializing SupportAgent...

--- Testing Retrieval for: ['mouse', 'keyboard', 'Razer BlackShark V2 Pro', 'FakeRazer 9000', 'Razer Basilisk V3 Pro'] ---

Product #1:
  - product_name: Razer DeathAdder V3 Pro
  - category: Mouse
  - description: Ergonomic wireless gaming mouse featuring a 30K optical sensor, 90 million click lifecycle optical switches, and a 63g ultra-lightweight design tailored for right-handed palm grips.
  - price: 149.99
  - pros: Extremely lightweight; Excellent sensor performance; Comfortable ergonomic shape.
  - cons: No RGB lighting; Only for right-handed users.
--------------------------------------------------
Product #2:
  - product_name: Razer Viper V2 Pro
  - category: Mouse
  - description: Symmetrical ultra-lightweight (58g) wireless esports mouse equipped with a Focus Pro 30K optical sensor, Gen-3 optical mouse switches, and up to 80 hours of battery life.
  - price: 149.99
  - pros: Flawless wireless tracking; Ambidextrous shape; Very low weight.
  - co